# Data Exploration: MC vs Run 3

Interactive notebook for exploring the feature distributions of:
- **MC** training data (`data/monte_carlo/training_data.h5`)
- **Run 3** real ATLAS data (loaded from `data/run3/file_3.root` via the NPZ cache)

Channel mapping (7 features per track):
| Ch | Name | Unit |
|---|---|---|
| 0 | d0 | mm |
| 1 | z0 | mm |
| 2 | d0_err | mm (always positive) |
| 3 | z0_err | mm (always positive) |
| 4 | d0_z0_cov | mm^2 |
| 5 | z_start | mm |
| 6 | z_end | mm |

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path
PROJECT_ROOT = Path().resolve().parent.parent.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import h5py
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 12})

print(f"Project root: {PROJECT_ROOT}")

## 1. Load MC Data

In [ ]:
MC_PATH = PROJECT_ROOT / "data" / "monte_carlo" / "training_data.h5"

with h5py.File(MC_PATH, "r") as f:
    print("Keys:", list(f.keys()))
    for key in f.keys():
        print(f"  {key}: shape={f[key].shape}, dtype={f[key].dtype}")

In [ ]:
# Load a subset of MC tracks for exploration
N_SUBEVENTS_TO_LOAD = 2400  # 200 events * 12 subevents
MASK_VAL = -999999.0

with h5py.File(MC_PATH, "r") as f:
    mc_tracks = f["tracks"][:N_SUBEVENTS_TO_LOAD]  # (N, 7, 695)

print(f"MC tracks shape: {mc_tracks.shape}")

# Extract valid (non-padded) tracks across all subevents
mc_d0 = []
mc_z0 = []
mc_d0_err = []
mc_z0_err = []
mc_cov = []

for i in range(mc_tracks.shape[0]):
    valid = mc_tracks[i, 1, :] > (MASK_VAL + 1)  # z0 channel as validity check
    if valid.sum() == 0:
        continue
    mc_d0.append(mc_tracks[i, 0, valid])
    mc_z0.append(mc_tracks[i, 1, valid])
    mc_d0_err.append(mc_tracks[i, 2, valid])
    mc_z0_err.append(mc_tracks[i, 3, valid])
    mc_cov.append(mc_tracks[i, 4, valid])

mc_d0 = np.concatenate(mc_d0)
mc_z0 = np.concatenate(mc_z0)
mc_d0_err = np.concatenate(mc_d0_err)
mc_z0_err = np.concatenate(mc_z0_err)
mc_cov = np.concatenate(mc_cov)

print(f"MC total valid tracks: {len(mc_d0):,}")

## 2. Load Run 3 Data (from file_3.root via NPZ cache)

In [ ]:
# The NPZ cache was pre-extracted from file_3.root
R3_CACHE = PROJECT_ROOT / "data" / "run3" / "cache_file3_2000ev_seed42.npz"

r3_data = np.load(R3_CACHE, allow_pickle=True)
print("Keys:", list(r3_data.keys()))
for key in r3_data.keys():
    arr = r3_data[key]
    print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")

In [ ]:
# Concatenate all Run 3 tracks (variable-length per event)
r3_d0 = np.concatenate(
    [np.asarray(x, dtype=np.float32) for x in r3_data["RecoTrack_d0"]]
)
r3_z0 = np.concatenate(
    [np.asarray(x, dtype=np.float32) for x in r3_data["RecoTrack_z0"]]
)
r3_d0_err = np.concatenate(
    [np.asarray(x, dtype=np.float32) for x in r3_data["RecoTrack_ErrD0"]]
)
r3_z0_err = np.concatenate(
    [np.asarray(x, dtype=np.float32) for x in r3_data["RecoTrack_ErrZ0"]]
)
r3_cov = np.concatenate(
    [np.asarray(x, dtype=np.float32) for x in r3_data["RecoTrack_ErrD0Z0"]]
)

print(f"Run 3 total tracks: {len(r3_d0):,}")
print(f"Run 3 total events: {len(r3_data['RecoTrack_z0'])}")

## 3. Basic Statistics

In [ ]:
import pandas as pd


def stats_row(name, arr):
    return {
        "feature": name,
        "mean": np.mean(arr),
        "std": np.std(arr),
        "min": np.min(arr),
        "median": np.median(arr),
        "max": np.max(arr),
        "N": len(arr),
    }


rows = []
for label, arrays in [
    ("MC", [mc_d0, mc_z0, mc_d0_err, mc_z0_err, mc_cov]),
    ("Run3", [r3_d0, r3_z0, r3_d0_err, r3_z0_err, r3_cov]),
]:
    for name, arr in zip(["d0", "z0", "d0_err", "z0_err", "d0_z0_cov"], arrays):
        row = stats_row(f"{label}/{name}", arr)
        rows.append(row)

df = pd.DataFrame(rows).set_index("feature")
df.style.format("{:.4f}", subset=["mean", "std", "min", "median", "max"])

## 4. Feature Distributions — MC vs Run 3

In [ ]:
features = [
    ("d0 [mm]", mc_d0, r3_d0, False),
    ("z0 [mm]", mc_z0, r3_z0, False),
    ("d0_err [mm]", mc_d0_err, r3_d0_err, True),
    ("z0_err [mm]", mc_z0_err, r3_z0_err, True),
    ("d0_z0_cov [mm^2]", mc_cov, r3_cov, False),
]

fig, axes = plt.subplots(3, 2, figsize=(12, 14))
axes = axes.flatten()

for idx, (label, mc_v, r3_v, use_log) in enumerate(features):
    ax = axes[idx]
    combined = np.concatenate([mc_v, r3_v])
    lo, hi = np.percentile(combined, [1, 99])
    bins = np.linspace(lo, hi, 80)

    ax.hist(mc_v, bins=bins, alpha=0.6, density=True, label="MC", color="#4c72b0")
    ax.hist(r3_v, bins=bins, alpha=0.6, density=True, label="Run 3", color="#dd8452")
    ax.axvline(np.mean(mc_v), color="#4c72b0", ls="--", lw=1)
    ax.axvline(np.mean(r3_v), color="#dd8452", ls="--", lw=1)
    if use_log:
        ax.set_yscale("log")
    ax.set_xlabel(label)
    ax.set_ylabel("Density")
    ax.legend()
    ax.grid(alpha=0.3, ls="--")

axes[-1].axis("off")
fig.suptitle("Feature Distributions: MC vs Run 3", fontsize=16)
fig.tight_layout()
plt.show()

## 5. 2D Correlations

In [ ]:
from matplotlib.colors import LogNorm

pairs = [
    ("z0", "d0", mc_z0, mc_d0, r3_z0, r3_d0),
    ("d0_err", "z0_err", mc_d0_err, mc_z0_err, r3_d0_err, r3_z0_err),
    ("d0", "d0_err", mc_d0, mc_d0_err, r3_d0, r3_d0_err),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for col, (xname, yname, mc_x, mc_y, r3_x, r3_y) in enumerate(pairs):
    all_x = np.concatenate([mc_x, r3_x])
    all_y = np.concatenate([mc_y, r3_y])
    x_lo, x_hi = np.percentile(all_x, [1, 99])
    y_lo, y_hi = np.percentile(all_y, [1, 99])
    extent = [x_lo, x_hi, y_lo, y_hi]

    for row, (dx, dy, label, cmap) in enumerate(
        [
            (mc_x, mc_y, "MC", "Blues"),
            (r3_x, r3_y, "Run 3", "Oranges"),
        ]
    ):
        ax = axes[row, col]
        hb = ax.hexbin(
            dx, dy, gridsize=50, cmap=cmap, extent=extent, mincnt=1, norm=LogNorm()
        )
        plt.colorbar(hb, ax=ax)
        ax.set_xlabel(xname)
        ax.set_ylabel(yname)
        ax.set_title(f"{label}: {xname} vs {yname}")

fig.suptitle("2D Feature Correlations", fontsize=16)
fig.tight_layout()
plt.show()

## 6. Per-Event Track Multiplicity

In [ ]:
# MC: count valid tracks per subevent
mc_trk_counts = []
for i in range(mc_tracks.shape[0]):
    valid = mc_tracks[i, 1, :] > (MASK_VAL + 1)
    mc_trk_counts.append(valid.sum())
mc_trk_counts = np.array(mc_trk_counts)

# Run 3: tracks per event
r3_trk_counts = np.array([len(np.asarray(x)) for x in r3_data["RecoTrack_z0"]])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# MC per-subevent
mc_nz = mc_trk_counts[mc_trk_counts > 0]
ax1.hist(mc_nz, bins=np.arange(0, mc_nz.max() + 2), alpha=0.7, color="#4c72b0")
ax1.set_xlabel("Tracks per sub-event")
ax1.set_ylabel("Count")
ax1.set_title(f"MC track multiplicity (median={np.median(mc_nz):.0f})")
ax1.grid(alpha=0.3, ls="--")

# Run 3 per-event
ax2.hist(r3_trk_counts, bins=50, alpha=0.7, color="#dd8452")
ax2.set_xlabel("Tracks per event")
ax2.set_ylabel("Count")
ax2.set_title(f"Run 3 track multiplicity (median={np.median(r3_trk_counts):.0f})")
ax2.grid(alpha=0.3, ls="--")

fig.tight_layout()
plt.show()

## 7. Beam Spot and z0 Shift

In [ ]:
beam_z = np.array([float(np.atleast_1d(x)[0]) for x in r3_data["BeamPosZ"]])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Beam spot distribution
ax1.hist(beam_z, bins=50, alpha=0.7, color="#6a3d9a")
ax1.axvline(
    np.mean(beam_z), color="red", ls="--", label=f"mean={np.mean(beam_z):.3f} mm"
)
ax1.set_xlabel("BeamPosZ [mm]")
ax1.set_ylabel("Count")
ax1.set_title("Run 3 Beam Spot Z")
ax1.legend()
ax1.grid(alpha=0.3, ls="--")

# z0 comparison zoomed
bins = np.linspace(-15, 15, 100)
ax2.hist(
    mc_z0,
    bins=bins,
    alpha=0.6,
    density=True,
    label=f"MC (mean={np.mean(mc_z0):.2f})",
    color="#4c72b0",
)
ax2.hist(
    r3_z0,
    bins=bins,
    alpha=0.6,
    density=True,
    label=f"Run3 (mean={np.mean(r3_z0):.2f})",
    color="#dd8452",
)
ax2.set_xlabel("z0 [mm]")
ax2.set_ylabel("Density")
ax2.set_title("z0 zoomed [-15, 15] mm")
ax2.legend()
ax2.grid(alpha=0.3, ls="--")

fig.tight_layout()
plt.show()

print(f"\nz0 shift (Run3 - MC): {np.mean(r3_z0) - np.mean(mc_z0):.3f} mm")
print(f"Beam spot mean z: {np.mean(beam_z):.3f} mm")

## 8. Explore file_3.root directly (if uproot available)

In [ ]:
ROOT_PATH = PROJECT_ROOT / "data" / "run3" / "file_3.root"

try:
    import uproot

    f = uproot.open(ROOT_PATH)
    print("Trees in file_3.root:")
    for key in f.keys():
        print(f"  {key}")

    # Show branches of the first tree
    first_tree = f[f.keys()[0]]
    print(f"\nBranches in {f.keys()[0]}:")
    for branch in first_tree.keys():
        print(f"  {branch}: {first_tree[branch].typename}")
    f.close()
except ImportError:
    print("uproot not installed. Install with: pip install uproot")
    print("Skipping direct ROOT file exploration.")
except Exception as e:
    print(f"Could not open ROOT file: {e}")